# TP 21 - Orchestration et outils : l'assistant NutriScope

## Objectif

Construire un assistant conversationnel multi-tours. Il répond aux questions d'un utilisateur sur un produit alimentaire et choisit lui-même l'outil à appeler : chercher le produit dans la base, proposer un produit de substitution mieux noté, ou expliquer en citant une source.

## Architecture

| Brique | Outil | Rôle |
|---|---|---|
| Agent, mémoire, outils | LangChain | fait tourner la boucle de l'agent et garde l'historique de chaque conversation |
| Modèle de chat | "ChatOpenAI" branché sur Ollama | comprend la question, choisit l'outil, rédige la réponse |
| Recherche documentaire | LlamaIndex, vecteurs stockés dans Chroma | retrouve les passages utiles et leur source |
| Suivi des conversations | journal JSONL et traces MLflow, en local | garde, pour chaque échange, les outils appelés, la durée et les tokens |

## Les outils de l'assistant

| Outil | Ce qu'il fait | Vrai code prévu | État dans ce notebook |
|---|---|---|---|
| "search_product" | trouve un produit par son nom ou son code-barres | TP 18 | placeholder |
| "suggest_substitutes" | propose des produits comparables mieux notés, en bio si demandé | TP 16 | placeholder |
| "answer_with_sources" | retrouve des passages de documents et donne leur source | TP 19-20 | placeholder, puis mini-index LlamaIndex |

Attention : les données renvoyées par les placeholders sont inventées. Elles respectent le format des vrais outils mais ne doivent jamais servir à une démonstration.

## Hors périmètre

- LangSmith : c'est un service en ligne, les conversations sortiraient de la machine. MLflow fait le même travail en local.
- Le résumé automatique de la mémoire et la confirmation humaine : pas nécessaires pour des conversations de quelques tours.
- La prédiction du Nutri-Score : elle deviendra un quatrième outil quand le modèle ML sera entraîné et sauvegardé.

## Choix des outils

- LangChain plutôt qu'une boucle écrite à la main avec le client "openai" : la boucle de l'agent, la déclaration des outils et la mémoire sont fournies, il reste moins de code à maintenir.
- "ChatOpenAI" plutôt que "ChatOllama" : c'est le même format OpenAI que l'atelier scanner. On passe d'Ollama en local à un fournisseur en ligne en changeant seulement l'adresse et le nom du modèle.
- LlamaIndex pour le RAG : le corpus sera fait de documents bruts de formats variés. LlamaIndex fournit leur lecture, leur découpage et leur réindexation, qu'il faudrait sinon écrire à la main format par format. Je veux aussi maîtriser cet outil pour des utilisations futures.
- MLflow pour tracer les conversations : les traces restent sur la machine, sans compte ni clé. Le même outil servira au suivi des entraînements ML. Il complète le journal JSONL, qui reste le journal rendu.

## Plan du notebook

1. Connexion au modèle
2. LangChain en bref
3. Le RAG avec LlamaIndex
4. Les outils "search_product" et "suggest_substitutes"
5. L'agent avec mémoire courte
6. Journalisation (journal JSONL et traces MLflow)
7. Scénarios rejouables
8. Bilan

## 1. Connexion au modèle

Le modèle de chat est "qwen3:8b", servi en local par Ollama. On l'appelle avec "ChatOpenAI", au format OpenAI, comme dans l'atelier scanner.

La température est à 0 : à question égale, le modèle répond de la même façon, ce qui rend les scénarios rejouables.

In [19]:
import os

import numpy as np

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding

In [9]:
load_dotenv()  # lit le fichier .env s'il existe

OLLAMA_URL = os.getenv("OLLAMA_URL") or "http://localhost:11434"
CHAT_MODEL = "qwen3:8b"

model = ChatOpenAI(
    base_url=f"{OLLAMA_URL}/v1",  # point d'accès au format OpenAI d'Ollama
    api_key="ollama",             # obligatoire pour le format OpenAI, ignorée par Ollama
    model=CHAT_MODEL,
    temperature=0,
)

In [10]:
response = model.invoke("Réponds en une phrase : qu'est-ce que le Nutri-Score ?")

print(response.content)
print(response.usage_metadata)

Le Nutri-Score est un système de notation des aliments, utilisé en France, qui classe les produits alimentaires sur une échelle de A (le plus sain) à E (le moins sain) en fonction de leur teneur en nutriments et en ingrédients peu bénéfiques pour la santé.
{'input_tokens': 27, 'output_tokens': 266, 'total_tokens': 293, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


## 2. LangChain en bref

Trois briques de base, avant de construire l'agent :

1. les messages : une conversation est une liste de messages, chacun avec un rôle
2. les gabarits de prompt : un texte à trous, rempli au moment de l'appel
3. les chaînes : des briques branchées les unes derrière les autres avec "|"

### 2.1. Les messages

Chaque message a un rôle. "SystemMessage" donne les consignes de l'assistant, "HumanMessage" porte la question de l'utilisateur, et la réponse du modèle est un "AIMessage".

In [11]:
messages = [
    SystemMessage("Tu es l'assistant NutriScope. Réponds en une phrase."),
    HumanMessage("Les chips sont-elles bonnes pour la santé ?"),
]

response = model.invoke(messages)
print(type(response).__name__, ":", response.content)

AIMessage : Les chips sont généralement peu saines en raison de leur teneur élevée en graisses, sel et calories, mais peuvent être consommées en modération ou remplacées par des alternatives plus saines comme les chips croustillantes faites maison.


### 2.2. Les gabarits de prompt

"ChatPromptTemplate" contient des variables entre accolades. Elles sont remplies à l'appel, et une variable oubliée provoque une erreur au lieu d'envoyer un prompt incomplet.

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es l'assistant NutriScope. Réponds en une phrase."),
    ("human", "Le produit {product} est-il bon pour la santé ?"),
])

print(prompt.invoke({"product": "Nutella"}).to_messages())

[SystemMessage(content="Tu es l'assistant NutriScope. Réponds en une phrase.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Le produit Nutella est-il bon pour la santé ?', additional_kwargs={}, response_metadata={})]


### 2.3. Les chaînes

L'opérateur "|" branche les briques : la sortie de l'une devient l'entrée de la suivante. Ici, le gabarit produit les messages, le modèle répond, et "StrOutputParser" ne garde que le texte de la réponse.

In [13]:
chain = prompt | model | StrOutputParser()

print(chain.invoke({"product": "Nutella"}))

Nutella est un produit sucré et gras, à consommer avec modération en raison de son taux élevé de sucre et de matières grasses, mais il apporte des nutriments comme la fibre et les antioxydants.


### 2.4. Le modèle ne se souvient de rien

Chaque appel est indépendant. Une question comme "Pourquoi ?", posée seule, n'a aucun sens pour le modèle. Pour qu'il suive la conversation, il faut lui renvoyer tout l'historique à chaque appel.

Ici on le fait à la main, avec une liste. Dans la section 5, "InMemorySaver" le fera automatiquement pour l'agent.

In [14]:
# Sans historique : le modèle ne sait pas de quoi on parle
print(model.invoke("Pourquoi ?").content)

Pourquoi ? 😊  
Pourriez-vous préciser de quoi vous parlez ? Je suis là pour vous aider !


In [15]:
# Avec historique : on renvoie toute la conversation à chaque appel
history = [
    SystemMessage("Tu es l'assistant NutriScope. Réponds en une phrase."),
    HumanMessage("Le Nutella est-il bon pour la santé ?"),
]
history.append(model.invoke(history))  # on garde la réponse dans l'historique
history.append(HumanMessage("Pourquoi ?"))

print(model.invoke(history).content)

Le Nutella est riche en sucre et en graisses (notamment en huile de palme), ce qui en fait un aliment calorique et peu nutritif, à consommer avec modération pour éviter les excès et préserver la santé.


## 3. Le RAG avec LlamaIndex

### 3.1. Le principe

Le modèle ne connaît pas nos sources et peut inventer (voir l'exemple du Nutella en 2.3). Le RAG (Retrieval-Augmented Generation, génération augmentée par la recherche) lui fournit des extraits de documents fiables au moment de répondre.

| Étape | Ce qu'elle fait | Quand |
|---|---|---|
| Lire | extraire le texte des pages web et des PDF de "data/corpus/" | une fois, à l'indexation |
| Découper | couper les longs textes en morceaux qui se recouvrent un peu | une fois, à l'indexation |
| Vectoriser | transformer chaque morceau en vecteur avec le modèle d'embeddings | une fois, à l'indexation |
| Ranger | stocker les vecteurs et leurs métadonnées dans Chroma | une fois, à l'indexation |
| Retrouver | vectoriser la question et chercher les morceaux les plus proches | à chaque question |

LlamaIndex s'occupe de ces cinq étapes. Il ne génère aucune réponse : les extraits retrouvés seront donnés à l'agent LangChain, qui rédigera la réponse en citant ses sources.

Le corpus compte 5 documents officiels (Vie publique, Santé.fr, Santé publique France, recommandations PNNS pour les adultes, avis du HCSP de 2017). Les fiches produits ne sont pas indexées ici : elles passent par l'outil "search_product".


### 3.2. Le modèle d'embeddings

Un embedding est un vecteur de nombres qui représente le sens d'un texte. Deux textes de sens proche donnent des vecteurs proches.

On utilise "qwen3-embedding:0.6b", servi par Ollama. Il est réglé une seule fois dans "Settings", les réglages globaux de LlamaIndex. Les documents et les questions passent ainsi par le même modèle, sinon les distances entre vecteurs n'auraient pas de sens.

Ce modèle attend une consigne devant les questions ("Instruct: ... Query:"), mais pas devant les documents. C'est le rôle de "query_instruction".


In [17]:
EMBED_MODEL = "qwen3-embedding:0.6b"

Settings.embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=OLLAMA_URL,  # sans "/v1" : LlamaIndex utilise l'API d'Ollama, pas le format OpenAI
    query_instruction="Instruct: Retrouver les passages qui répondent à une question sur la nutrition\nQuery:",
)

vector = Settings.embed_model.get_query_embedding("Comment est calculé le Nutri-Score ?")
print("Dimension du vecteur :", len(vector))

Dimension du vecteur : 1024


Les nombres d'un vecteur n'ont pas de sens pris un par un. Ce qui compte, c'est la comparaison entre deux vecteurs, mesurée par la similarité cosinus : proche de 1 pour des sens proches, plus basse pour des textes sans rapport.

On compare la question à quatre phrases : une qui y répond, une fausse qui a la même forme, une qui parle du Nutri-Score sans y répondre, et une sur un autre sujet.


In [21]:
def cosine_similarity(a, b):
    """Calcule la similarité cosinus entre deux vecteurs (1 = même direction)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


question = "Comment est calculé le Nutri-Score ?"
sentences = [
    "Le Nutri-Score est calculé pour 100 g : l'énergie, les sucres, les graisses saturées et le sel font baisser la note, les fibres, les protéines, les fruits et les légumes la font monter.",
    "Le Nutri-Score est calculé à partir du prix du produit et de sa marque.",  # phrase fausse
    "Le Nutri-Score est affiché sur les emballages de façon volontaire par les fabricants.",
    "Il est recommandé de limiter la charcuterie à 150 g par semaine.",
]

question_vector = Settings.embed_model.get_query_embedding(question)  # avec la consigne

for sentence in sentences:
    sentence_vector = Settings.embed_model.get_text_embedding(sentence)  # sans la consigne
    score = cosine_similarity(question_vector, sentence_vector)
    print(f"{score:.2f}  {sentence[:70]}...")

0.86  Le Nutri-Score est calculé pour 100 g : l'énergie, les sucres, les gra...
0.78  Le Nutri-Score est calculé à partir du prix du produit et de sa marque...
0.63  Le Nutri-Score est affiché sur les emballages de façon volontaire par ...
0.28  Il est recommandé de limiter la charcuterie à 150 g par semaine....


La phrase fausse obtient 0,78, presque autant que la bonne réponse (0,86) : elle a le même sujet et la même forme. La similarité mesure la proximité de sens, pas la vérité. D'où l'importance d'un corpus de sources fiables, de plusieurs extraits renvoyés (top-k) et d'une mesure de la recherche.
